In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-DfzPoQJoPpxfkAFCCvqPux-ceb22bBsHMgJAtzUkxqkwq")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['UCL/mimix','BioMedIA/MIRTK','RaymanNg/3D-Wind-Field','kick-h/gpumd','mastoffel/rptR','conda-forge/pyinterp-feedstock','shader-slang/slang','theislab/scib','WorldWideTelescope/wwt-web-client','ANTsX/ANTsR']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

In [ ]:
df = pd.read_csv("df_commit_data.csv")

summary = df[["project", "commit", "author", "author_time", "message"]].copy()

summary.columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

summary.to_csv(
    "kyandall_project_summary.csv",
    sep=";",
    index=False
)

print(summary.shape)
summary.head()

In [ ]:
import pandas as pd
import time
from tqdm import tqdm


df = pd.read_csv("df_commits.csv")


df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

commit_data = []

chunks = [
    df["sha1"].iloc[i:i+10]
    for i in range(0, len(df), 10)
]

for chunk in tqdm(chunks):
    res, err = woc.get_values_many(
        "commit.tch",
        chunk.tolist()
    )

    res = {k: v[0] for k, v in res.items()}

    for commit_sha, commit in res.items():
        commit_data.append({
            "commit": commit_sha,
            "author": commit[2][0],
            "author_time": int(commit[2][1]),
            "message": commit[4]
        })

    if err:
        print("Errors:", err)

    time.sleep(1)

df_commit_data = pd.DataFrame(commit_data)

df_commit_data = df_commit_data.merge(
    df[["sha1", "project"]],
    left_on="commit",
    right_on="sha1",
    how="left"
)


df_commit_data.to_csv(
    "df_commit_data.csv",
    index=False
)

print("DONE")
print("Rows:", len(df_commit_data))
print(df_commit_data.head())

In [ ]:
print(len(commit_data))

In [ ]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
import pandas as pd
import time
from tqdm import tqdm

# Load the commit SHA/project list you already generated
df = pd.read_csv("df_commits.csv")

# Your df_commits.csv may have an old pandas index column
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

chunks = [df["sha1"][x:x+10] for x in range(0, len(df), 10)]

commit_data = []

for chunk in tqdm(chunks):
    res, err = woc.get_values_many("commit.tch", chunk.to_list())

    res = {k: v[0] for k, v in res.items()}

    if err:
        print("Got Errors:", err)

    for commit_sha, commit in res.items():
        commit_data.append({
            "commit": commit_sha,
            "author": commit[2][0],
            "author_time": int(commit[2][1]),
            "message": commit[4],
        })

    time.sleep(1)

# Turn retrieved information into dataframe
df_commit_data = pd.DataFrame(commit_data)

# Add the correct project to every commit
df_commit_data = df_commit_data.merge(
    df[["sha1", "project"]],
    left_on="commit",
    right_on="sha1",
    how="left"
)

# SAVE THE RAW RETRIEVED DATA
df_commit_data.to_csv("df_commit_data.csv", index=False)

print("Created df_commit_data.csv")
print(df_commit_data.shape)

df_commit_data.head()

In [ ]:
df_commit_data = pd.read_csv("df_commit_data_partial.csv")

summary = df_commit_data[
    ["project", "commit", "author", "author_time", "message"]
].copy()

summary.columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

summary.to_csv(
    "kyandall_project_summary.csv",
    sep=";",
    index=False
)

print("DONE:", len(summary))

In [16]:
import pandas as pd

summary = pd.read_csv(
    "kyandall_project_summary.csv",
    sep=";"
)

woc_stats = summary.groupby("project_wocid").agg(
    num_commits=("commit_sha1", "count"),
    num_authors=("author", "nunique"),
    min_time=("time", "min"),
    max_time=("time", "max")
)

woc_stats

,num_commits,num_authors,min_time,max_time
project_wocid,,,,
antsx_antsr,5482,45,1328136237,1762619138
biomedia_mirtk,2568,22,1450392347,1741992251
conda-forge_pyinterp-feedstock,456,16,1568666567,1761982417
kick-h_gpumd,5936,81,1500046377,1760973080
mastoffel_rptr,431,11,1414329721,1745921255
raymanng_3d-wind-field,267,8,1547622404,1750236118
shader-slang_slang,29535,317,1497030169,1761660359
theislab_scib,2253,60,1563266409,1761287409
ucl_mimix,563,6,1564409694,1636040514


In [17]:
woc_stats["min_date"] = pd.to_datetime(
    woc_stats["min_time"],
    unit="s"
)

woc_stats["max_date"] = pd.to_datetime(
    woc_stats["max_time"],
    unit="s"
)

woc_stats

,num_commits,num_authors,min_time,max_time,min_date,max_date
project_wocid,,,,,,
antsx_antsr,5482,45,1328136237,1762619138,2012-02-01 22:43:57,2025-11-08 16:25:38
biomedia_mirtk,2568,22,1450392347,1741992251,2015-12-17 22:45:47,2025-03-14 22:44:11
conda-forge_pyinterp-feedstock,456,16,1568666567,1761982417,2019-09-16 20:42:47,2025-11-01 07:33:37
kick-h_gpumd,5936,81,1500046377,1760973080,2017-07-14 15:32:57,2025-10-20 15:11:20
mastoffel_rptr,431,11,1414329721,1745921255,2014-10-26 13:22:01,2025-04-29 10:07:35
raymanng_3d-wind-field,267,8,1547622404,1750236118,2019-01-16 07:06:44,2025-06-18 08:41:58
shader-slang_slang,29535,317,1497030169,1761660359,2017-06-09 17:42:49,2025-10-28 14:05:59
theislab_scib,2253,60,1563266409,1761287409,2019-07-16 08:40:09,2025-10-24 06:30:09
ucl_mimix,563,6,1564409694,1636040514,2019-07-29 14:14:54,2021-11-04 15:41:54


# Part 1 Project Summary

## antsx_antsr

### GitHub
- Stars: 143
- Forks: 32
- Last commit/update date: 2026-06-12

### WoC
- Number of commits: 5482
- Number of authors: 45
- Min time: 1328136237
- Max time: 1762619138
- Min date: 2012-02-01 22:43:57
- Max date: 2025-11-08 16:25:38

## biomedia_mirtk

### GitHub
- Stars: 200
- Forks: 75
- Last commit/update date: 2025-02-27

### WoC
- Number of commits: 2568
- Number of authors: 22
- Min time: 1450392347
- Max time: 1741992251
- Min date: 2015-12-17 22:45:47
- Max date: 2025-03-14 22:44:11

## conda-forge_pyinterp-feedstock

### GitHub
- Stars: 0
- Forks: 4
- Last commit date: 2025-11-01

### WoC
- Number of commits: 456
- Number of authors: 16
- Min time: 1568666567
- Max time: 1761982417
- Min date: 2019-09-16 20:42:47
- Max date: 2025-11-01 07:33:37

## kick-h_gpumd

### GitHub
- Stars: 2
- Forks: 1
- Last commit date: 2025-10-20

### WoC
- Number of commits: 5936
- Number of authors: 81
- Min time: 1500046377
- Max time: 1760973080
- Min date: 2017-07-14 15:32:57
- Max date: 2025-10-20 15:11:20

## mastoffel_rptr

### GitHub
- Stars: 19
- Forks: 1
- Last commit date: 2025-04-29

### WoC
- Number of commits: 431
- Number of authors: 11
- Min time: 1414329721
- Max time: 1745921255
- Min date: 2014-10-26 13:22:01
- Max date: 2025-04-29 10:07:35

## raymanng_3d-wind-field

### GitHub
- Stars: 497
- Forks: 159
- Last commit/update date: 2025-06-18

### WoC
- Number of commits: 267
- Number of authors: 8
- Min time: 1547622404
- Max time: 1750236118
- Min date: 2019-01-16 07:06:44
- Max date: 2025-06-18 08:41:58

## shader-slang_slang

### GitHub
- Stars: 5.7k
- Forks: 496
- Last commit/update date: 2026-09-27

### WoC
- Number of commits: 29535
- Number of authors: 317
- Min time: 1497030169
- Max time: 1761660359
- Min date: 2017-06-09 17:42:49
- Max date: 2025-10-28 14:05:59

## theislab_scib

### GitHub
- Stars: 432
- Forks: 78
- Last commit/update date: 2026-04-27

### WoC
- Number of commits: 2253
- Number of authors: 60
- Min time: 1563266409
- Max time: 1761287409
- Min date: 2019-07-16 08:40:09
- Max date: 2025-10-24 06:30:09

## ucl_mimix

### GitHub
- Current repository: UCL/RefBasedMI
- Stars: 5
- Forks: 4
- Last commit/release-era date: 2024-09-04

### WoC
- Number of commits: 563
- Number of authors: 6
- Min time: 1564409694
- Max time: 1636040514
- Min date: 2019-07-29 14:14:54
- Max date: 2021-11-04 15:41:54

## worldwidetelescope_wwt-web-client

### GitHub
- Stars: 108
- Forks: 35
- Last commit/update date: 2026-06-20

### WoC
- Number of commits: 1416
- Number of authors: 24
- Min time: 1416030738
- Max time: 1753308292
- Min date: 2014-11-15 05:52:18
- Max date: 2025-07-23 22:04:52

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github